# Short-Term Memory

본 실험은 대화의 세션별 저장·조회, 의미 기반 검색, 데이터 삭제의 동작을 관찰한다.
개념적 배경은 [00. Agent Memory: 개념과 동작 구조](00_agent_memory_overview.ipynb)에 제시한다.

하나의 `MemoryClient`를 준비 단계에서 연결하여 모든 작업 셀에서 재사용한다. 실험 종료 시 세션 데이터 정리와 연결 종료를 별도로 수행한다.

## 준비

로그 출력 수준, 메모리 설정, 클라이언트 연결, 세션 식별자를 구성한다.
이후 실험 셀은 준비 단계에서 생성한 `settings`, `memory_client`, `session_id`를 공유한다.

### 로그 출력 설정

Neo4j 서버 알림과 Transformers 모델 설정 모듈의 로그를 `ERROR` 이상으로 제한한다.
이 설정은 경고의 표시 여부를 제어하며, 경고의 원인이나 실행 중 예외를 처리하지 않는다.
설정은 현재 커널에 적용되며 커널 재시작 후 다시 실행해야 한다.

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

### 환경 변수와 메모리 설정

`load_memory_settings()`는 `.env`와 프로젝트 설정을 읽어 `MemorySettings` 객체를 생성한다.
설정 객체는 Neo4j 접속 정보, 임베딩 제공자, 추출 파이프라인 구성을 포함한다.
`print_memory_settings()`는 적용할 추출기와 모델 설정을 출력한다.

이 단계는 설정 구성에 해당한다. DB 연결과 스키마 초기화는 다음 셀에서 수행한다.

In [ ]:
from config.memory_client import load_memory_settings, print_memory_settings
from neo4j_agent_memory import MemorySettings

settings: MemorySettings = load_memory_settings()
print_memory_settings(settings)

### 메모리 클라이언트 연결

`MemoryClient(settings)`로 객체를 생성하고 `connect()`를 호출하여 Neo4j 연결과 스키마·메모리 구성 요소를 초기화한다.
이후 저장·조회·검색·삭제 셀은 동일한 `memory_client`를 사용한다.
연결 과정에서 예외가 발생하면 `close()`로 자원을 정리한 뒤 예외를 다시 발생시킨다.

이 셀은 실행할 때마다 새 클라이언트를 생성한다. 재실행 전에는 기존 클라이언트의 종료가 선행되어야 한다.
설정 변경 시에는 기존 연결 종료 → 설정 재구성 → 클라이언트 연결 순서로 수행한다.

In [ ]:
from neo4j_agent_memory import MemoryClient

memory_client = MemoryClient(settings)

try:
    await memory_client.connect()
except:
    await memory_client.close()
    raise

print("✓ Memory client opened")

### 세션 식별자 정의

`session_id`는 대화에 속한 메시지를 묶는 논리적 식별자이다.
**본 실험은 `01_short_term_memory`를 메시지 저장, 세션 이력 조회, 세션 삭제의 기준으로 사용한다.**

식별자 변경은 기존 데이터 삭제를 의미하지 않는다. 또한 현재 Bolt 구현의 의미 기반 검색은 다른 세션의 메시지를 포함할 수 있으므로, 세션별 이력 조회와 검색의 범위를 구별해야 한다.

In [ ]:
session_id = "01_short_term_memory"

## 실험

메시지 저장, 저장 그래프 시각화, 세션별 이력 조회, 질의 기반 컨텍스트 검색을 순서대로 수행한다.
저장된 메시지 수와 반환된 컨텍스트 구역을 통해 각 연산의 동작 범위를 관찰한다.

### 대화 메시지 저장

예제 문장 4개를 현재 세션에 `user` 역할로 순차 저장한다.
현재 설정과 기본 인자에서는 `add_message()`가 메시지 임베딩을 생성하고, 메시지 저장 후 추출 파이프라인을 실행하여 엔티티와 관계를 영속화한다.
각 호출의 반환값으로 메시지 ID를 확인한다.

동일 셀의 재실행은 동일 원문을 가진 별도 메시지를 추가한다. 중복 없는 반복 실험을 수행하려면 저장 전에 대상 세션의 이력을 정리해야 한다.
자동 추출을 제외한 저장 동작은 `extraction_mode="skip"` 조건으로 비교할 수 있다.

In [ ]:
messages = [
            "이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다.",
            "화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요.",
            "11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다.",
            "오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다.",
    ]

for message in messages:
    response = await memory_client.short_term.add_message(
        session_id=session_id,
        role="user",
        content=message
    )
    print(f"✓ Stored message: {response.id}")
    print(f"  [{message}]")

### Neo4j 저장 그래프 시각화

현재 `session_id`에 해당하는 대화를 시작점으로, 관계 1~3개로 연결된 경로를 최대 300개 조회한다.
Neo4j의 Python 시각화 라이브러리인 `neo4j-viz`가 조회 결과를 변환하고 노트북에 그래프 위젯으로 표시한다.
노드와 관계의 속성을 확인하고, 확대·축소 및 레이아웃 변경으로 저장 구조를 관찰한다.

기본 캡션은 노드의 레이블이므로, 저장 내용을 식별할 수 있도록 엔티티는 `name`, 메시지는 `content`, 대화는 `session_id`를 표시한다.
긴 캡션은 80자 이내로 축약하며, 원문은 노드의 상세 속성에 유지한다. 관계는 별도의 `type` 속성이 있으면 해당 값을 표시하고, 없으면 관계 타입을 표시한다.

`memory_client.graph.session()`으로 기존 Bolt 연결을 재사용한다. 조회 결과는 노드·관계 정보를 보존하는 `Graph`로 가져온 뒤 `from_neo4j()`에 전달한다.
조회용 세션은 `async with` 블록 종료 시 닫히며, 메모리 클라이언트 연결은 이후 실험에서도 유지된다.

`[*1..3]`은 관계 종류와 방향을 제한하지 않는다. 따라서 공유 엔티티를 통해 다른 세션의 메시지도 포함될 수 있다.
`LIMIT 300`은 반환 경로 수의 상한이며, 노드 수나 메시지 수의 상한이 아니다. 조회 결과가 없으면 빈 상태를 출력한다.

[Neo4j 쿼리 결과 가져오기](https://neo4j.com/docs/python-graph-visualization/current/integration/neo4j/) · [그래프 렌더링](https://neo4j.com/docs/python-graph-visualization/current/rendering/)


In [ ]:
from typing import LiteralString

from IPython.display import display
from neo4j_viz.neo4j import from_neo4j

graph_query: LiteralString = """
MATCH (c:Conversation {session_id: $session_id})
MATCH path = (c)-[*1..3]-(n)
RETURN path
LIMIT 300
"""

async with memory_client.graph.session() as graph_session:
    result = await graph_session.run(graph_query, session_id=session_id)
    graph = await result.graph()

if graph.nodes:
    visualization = from_neo4j(graph)
    for node in visualization.nodes:
        props = node.properties
        caption = str(props.get("name") or props.get("content") or props.get("session_id") or node.caption)
        node.caption = caption if len(caption) <= 80 else caption[:79] + "…"
    for relationship in visualization.relationships:
        relationship.caption = relationship.properties.get("__type") or relationship.caption
    display(visualization.render_widget(height="600px"))
else:
    print("시각화할 그래프가 없다. 메시지 저장 후 다시 조회한다.")


### 세션별 대화 이력 조회

`get_conversation(session_id=...)`으로 지정 세션의 대화를 조회하고, 반환된 메시지 수를 출력한다.
이 연산의 선택 기준은 세션 식별자이며, 검색어와의 의미적 유사도를 사용하지 않는다.

빈 세션에 예제 메시지 4개를 한 번 저장한 경우 예상 반환 수는 4개이다.
출력값은 이번 호출로 반환된 메시지 수이며, 대규모 대화에서는 조회 제한의 영향을 받을 수 있다.

In [ ]:
conversation = await memory_client.short_term.get_conversation(session_id=session_id)
print(f"✓ Retrieved {len(conversation.messages)} message(s)")

### 질의 기반 컨텍스트 검색

각 검색어에 대해 `get_context()`를 호출하여 후속 LLM 프롬프트에 사용할 컨텍스트 문자열을 구성한다.
이 셀의 연산 범위는 검색과 컨텍스트 출력이며, 별도의 LLM 응답 생성은 포함하지 않는다.

| 인자 | 적용 내용 |
|---|---|
| `include_short_term=True` | 현재 세션의 최근 이력과 의미적으로 관련된 과거 메시지를 포함한다. |
| `include_long_term=True` | 관련 엔티티와 저장된 선호도를 포함한다. |
| `include_reasoning=False` | 과거 작업의 추론 기록을 제외한다. |

현재 SDK의 Bolt 구현에서 `Recent Conversation`은 지정 세션에 속하지만, `Relevant Past Messages`는 전체 세션의 메시지를 대상으로 검색한다.
`Relevant Knowledge`도 현재 세션에 한정되지 않는다. 따라서 현재 세션 삭제 후에도 다른 세션의 메시지나 장기 기억이 반환될 수 있다. [검색 범위](https://neo4j.com/labs/agent-memory/reference/api/short-term-search/)

관련 항목이 없으면 해당 구역은 출력되지 않는다. 출력된 유사도는 관련성 점수이며 사실성에 대한 확률이 아니다.

In [ ]:
queries = ["오라클", "GCP", "사용량", "Cloud DNS"]

for query in queries:
    print(f"✓ Search query: [{query}]")

    context = await memory_client.get_context(
        query = query,
        session_id = session_id,
        include_short_term=True,
        include_long_term=True,
        include_reasoning=False,
    )

    print("Context:")
    print(context + "\n")

## 정리

세션 데이터 삭제와 클라이언트 연결 종료를 구분하여 수행한다.
대화 이력을 보존하는 경우에는 삭제 단계를 생략하고 연결 종료만 수행한다.

### 현재 세션의 대화 삭제

`clear_session(session_id=...)`으로 지정 세션의 대화와 연결된 메시지를 삭제한다.
삭제 후 같은 세션을 조회하여 `Remaining messages: 0`인지 확인한다.

검증 범위는 현재 세션의 대화 이력이다. 다른 세션의 메시지와 공유 엔티티·선호도 등의 장기 기억은 유지된다.
이 단계는 데이터 변경을 수반하므로, 실험 기록을 보존하는 경우 생략한다.

In [ ]:
await memory_client.short_term.clear_session(session_id=session_id)

conversation = await memory_client.short_term.get_conversation(session_id=session_id)
print(f"✓ Cleared conversation history: {session_id}")
print(f"  Remaining messages: {len(conversation.messages)}")

### 메모리 클라이언트 종료

`close()`로 클라이언트의 DB 연결과 백그라운드 작업을 정리한다.
연결 종료는 DB에 저장된 메시지나 장기 기억을 삭제하지 않는다.

종료 셀은 자동으로 실행되지 않는다. 정상 종료 또는 중간 오류로 실험을 마치는 시점에 명시적으로 실행해야 한다.
종료 이후 작업을 재개하려면 클라이언트 연결 단계부터 다시 수행한다.

In [ ]:
await memory_client.close()
print("✓ Memory client closed")